In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf chromadb sentence-transformers

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World",
    metadata={"source":"https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
#text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15388\3596417266.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
TextDocument = loader.load()

In [8]:
TextDocument

[Document(metadata={'source': 'data/python.txt'}, page_content='')]

In [9]:
# #PDF data
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("data/research.pdf")

# PDF_document = loader.load()

# PDF_document

## ingestion pipeline

In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader, PyMuPDFLoader

### Documents

In [11]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyMuPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print(f"total docs = {num_docs}")
    print(f"total pages = {len(all_docs)}")
    return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

total docs = 34
total pages = 229


In [13]:
type(all_pdf_documents[5])

langchain_core.documents.base.Document

### Chunks

In [14]:
#chunks
# !pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(document, chunk_size=500, chunk_overlap=50): # chunk_size means what is the maximuum num of charcter that each chunk is going to have
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(document)
    return chunked_docs

In [16]:
chunks = split_docs(all_pdf_documents)

In [17]:
len(chunks)

847

### Embedding

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
# os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxx" #to avoid the warning: You are sending unauthenticated requests to the HF Hub.

In [23]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimensions", self.model.get_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding.....", embeddings.shape)
        return embeddings

In [24]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions 384


### Vector store

In [25]:
import chromadb
import uuid  #help us to create idx

In [26]:
class VectorStoreManager:

    def __init__(
        self,
        persist_direcctory="data/vector_store",
        collection_name="pdf_documents"
    ):
        self.persist_direcctory = persist_direcctory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):

        os.makedirs(
            self.persist_direcctory,
            exist_ok=True
        )

        # Create Chroma client
        self.client = chromadb.PersistentClient(
            path=self.persist_direcctory
        )

        # Create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embeddings in RAG",
                "hnsw:space": "cosine"
            }
        )

        print(
            "Initialized vector store:",
            self.collection_name
        )

        print(
            "Documents in collection:",
            self.collection.count()
        )

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents does not match number of embeddings"
            )

        # Lists that will be sent to Chroma
        ids = []
        documents_content = []
        embeddings_list = []
        all_metadata = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            doc_id = f"doc_{uuid.uuid4()}"

            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            all_metadata.append(metadata)

            documents_content.append(
                doc.page_content
            )

            embeddings_list.append(
                embedding.tolist()
            )
            
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print(
            "Total documents added:",
            len(documents_content)
        )

        print(
            "Documents in collection:",
            self.collection.count()
        )

In [27]:
vector_store = VectorStoreManager()

Initialized vector store: pdf_documents
Documents in collection: 0


In [28]:
# dara => documents => chunks => embeddings => store in vector store
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

embedding..... (847, 384)
Total documents added: 847
Documents in collection: 847


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.4):
        #query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs = []
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieve {len(retrieved_docs)} documents")
            
        else:
            print("no document found...")

        return retrieved_docs

In [31]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [32]:
rag_retriever.retrieve("What is git")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 5 documents


[{'metadata': {'format': 'PDF 1.7',
   'source': 'data/pdfs\\research8.pdf',
   'moddate': '2026-06-01T18:15:36+00:00',
   'content_length': 64,
   'producer': 'pdfcpu v0.12.1 dev',
   'title': 'git-cheat-sheet-education',
   'trapped': '',
   'total_pages': 2,
   'creator': 'Adobe Illustrator CC (Macintosh)',
   'creationdate': '2026-06-01T18:15:36+00:00',
   'subject': '',
   'keywords': '',
   'page': 0,
   'creationDate': "D:20260601181536+00'00'",
   'file_path': 'data/pdfs\\research8.pdf',
   'modDate': "D:20260601181536+00'00'",
   'doc_index': 834,
   'author': ''},
  'document': 'Git for All Platforms\nhttp://git-scm.com\nnileshgale520@gmail.com',
  'distance': 0.381225049495697,
  'similarity_score': 0.618774950504303,
  'rank': 1},
 {'metadata': {'creator': 'Adobe Illustrator CC (Macintosh)',
   'source': 'data/pdfs\\research8.pdf',
   'modDate': "D:20260601181536+00'00'",
   'producer': 'pdfcpu v0.12.1 dev',
   'subject': '',
   'creationDate': "D:20260601181536+00'00'",
  

### NVIDIA API KEY

In [33]:

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv() # to load api key from .env

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found. Check your .env file.")

llm = ChatOpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
    model="meta/llama-3.1-70b-instruct",
    temperature=0.1,
    max_tokens=1024
)

In [34]:
def generate_output(query, retriever, llm, top_k=3):

    results = retriever.retrieve(query, top_k)

    context = "\n\n".join(
        [doc["document"] for doc in results]
    ) if results else ""

    if not context:
        return "I could not find relevant information in the provided documents."

    prompt = f"""
        You are a retrieval-augmented question answering assistant.
        
        Answer the question using ONLY the provided context.
        
        Rules:
        1. Do not use outside knowledge.
        2. Do not invent or assume information.
        3. If the answer is not present in the context, say:
           "I could not find the answer in the provided documents."
        4. Give a concise and direct answer.
        
        Context:
        {context}
        
        Question:
        {query}
        
        Answer:
        """

    response = llm.invoke(prompt)
    return response.content

In [42]:
answer = generate_output(
    "What is Logistic regression",
    rag_retriever,
    llm
)

print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding..... (1, 384)
retrieve 3 documents
Logistic Regression is a supervised ML algorithm for classification problems that predicts the probability that an input belongs to a specific class or category.


### OPENAI API

In [ ]:
# API_KEY_OPENAI = ""

In [ ]:
# !pip install -U langchain-openai

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(
#     openai_api_key = API_KEY_OPENAI,
#     model="gpt-5.4",
#     temperature=0.1,
#     max_token=1024
# )

In [ ]:
# def generate_output(query, retriever, llm, top_k=3):
#     results = retriever.retrieve(query, top_k)

#     context = "\n".join([doc["document"] for doc in results]) if results else ""

#     if not context:
#         print("we found no relevant context for the given query")

#     #context + query
#     prompt = f""" use given context to generate the answer for the query
#                 Context: {context}
#                 Query : {query}"""

#     response = llm.invoke(prompt) #expecting a string as prompt

#     return response.content

In [ ]:
# answer = generate_output("What is RAG", rag_retriever, llm)
# print(answer)

### GROQ

In [ ]:
# API_KEY_GROQ = ""

In [44]:
# !pip install langchain-groq

In [ ]:
# from langchain_groq import ChatGroq

# llm = ChatOpenAI(
#      groq_api_key = API_KEY_GROQ,
#      model="qwen/qwen3-32b",
#      temperature=0.1,
#      max_token=1024
# )

In [ ]:
# def generate_output(query, retriever, llm, top_k=3):
#     results = retriever.retrieve(query, top_k)

#     context = "\n".join([doc["document"] for doc in results]) if results else ""

#     if not context:
#         print("we found no relevant context for the given query")

#     #context + query
#     prompt = f""" use given context to generate the answer for the query
#                 Context: {context}
#                 Query : {query}"""

#     response = llm.invoke([prompt,format(context=context, query=query)]) #expecting a string as prompt

#     return response.content

In [ ]:
# answer = generate_output("What is RAG", rag_retriever, llm)
# print(answer)